In [5]:
%pip install ollama

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ollama]━━━━ 4/5 [ollama]c]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/nineleaps/PythonSession/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import ollama
response = ollama.generate(model="llama3.2:3b", prompt="Say hello in one sentence.")
print(response["response"])

Hello!


In [9]:
%pip install pdfplumber sentence-transformers chromadb ollama langchain-text-splitters

  Using cached pdfplumber-0.11.10-py3-none-any.whl.metadata (43 kB)
  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached pdfminer_six-20260107-py3-none-any.whl.metadata (4.3 kB)
  Using cached pypdfium2-5.13.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (66 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pybase64-1.5.0-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (11 kB)
  Using cached uvicorn-0.52.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.29.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metad

In [10]:
import pdfplumber
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import ollama

print("Imports OK")

/home/nineleaps/PythonSession/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [12]:
# ---- PDF SOURCE ----
PDF_PATH = "/home/nineleaps/Downloads/houseplant_care_guide.pdf"   # <-- CHANGE THIS to your PDF's filename/path

# ---- CHUNKING ----
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# ---- EMBEDDING ----
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ---- VECTOR STORAGE ----
DB_PATH = "./chroma_db_pdf"
COLLECTION_NAME = "pdf_docs"

# ---- RETRIEVAL ----
TOP_K = 4
MIN_SIMILARITY_THRESHOLD = 0.25

# ---- GENERATION ----
LLM_MODEL = "llama3.2:3b"

print(f"Config loaded. Target PDF: {PDF_PATH}")

Config loaded. Target PDF: /home/nineleaps/Downloads/houseplant_care_guide.pdf


In [13]:
def extract_pdf_text(pdf_path):
    """Extract text from every page of the PDF, tagged with page number."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({"page_number": i + 1, "text": text})
    return pages

raw_pages = extract_pdf_text(PDF_PATH)
print(f"Extracted text from {len(raw_pages)} pages.")
print("\n--- Preview of page 1 ---\n")
print(raw_pages[0]["text"][:500] if raw_pages else "(no text extracted)")

Extracted text from 3 pages.

--- Preview of page 1 ---

A Beginner's Guide to Houseplant Care
This guide covers the basics of keeping common houseplants healthy, including watering, light, soil,
and common problems.
Watering Basics
Most common houseplants should be watered when the top inch of soil feels dry to the touch.
Overwatering is the most common cause of houseplant death, as it leads to root rot. As a general rule,
water thoroughly until it drains from the bottom of the pot, then wait until the soil is dry again before
watering next.
The Snak


In [15]:
## 4. Text Cleaning
##Handles hyphenated line-breaks, repeated whitespace, excessive blank lines.

def clean_text(text):
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for p in raw_pages:
    p["text"] = clean_text(p["text"])

print("Cleaning complete.")
print(raw_pages[0]["text"][:500] if raw_pages else "")

Cleaning complete.
A Beginner's Guide to Houseplant Care
This guide covers the basics of keeping common houseplants healthy, including watering, light, soil,
and common problems.
Watering Basics
Most common houseplants should be watered when the top inch of soil feels dry to the touch.
Overwatering is the most common cause of houseplant death, as it leads to root rot. As a general rule,
water thoroughly until it drains from the bottom of the pot, then wait until the soil is dry again before
watering next.
The Snak


In [17]:
## 5. Chunking

def chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for page in pages:
        pieces = splitter.split_text(page["text"])
        for i, piece in enumerate(pieces):
            chunks.append({
                "id": f"page{page['page_number']}_chunk{i}",
                "text": piece,
                "page": page["page_number"],
            })
    return chunks

all_chunks = chunk_pages(raw_pages)
print(f"Produced {len(all_chunks)} chunks from {len(raw_pages)} pages.")
print(f"\nExample chunk (page {all_chunks[0]['page']}):\n{all_chunks[0]['text'][:300]}")

Produced 9 chunks from 3 pages.

Example chunk (page 1):
A Beginner's Guide to Houseplant Care
This guide covers the basics of keeping common houseplants healthy, including watering, light, soil,
and common problems.
Watering Basics
Most common houseplants should be watered when the top inch of soil feels dry to the touch.
Overwatering is the most common 


In [18]:
## 6. Embedding

embed_model = SentenceTransformer(EMBEDDING_MODEL)

texts = [c["text"] for c in all_chunks]
print(f"Embedding {len(texts)} chunks...")
embeddings = embed_model.encode(texts, show_progress_bar=True).tolist()
print(f"Done. Each embedding has {len(embeddings[0])} dimensions.")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 1080.30it/s]


Embedding 9 chunks...


Batches: 100%|███████████████████████████████████| 1/1 [00:00<00:00,  1.85it/s]

Done. Each embedding has 384 dimensions.


In [19]:
## 7. Vector Storage

client = chromadb.PersistentClient(path=DB_PATH)

try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(COLLECTION_NAME)

collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=embeddings,
    documents=texts,
    metadatas=[{"page": c["page"]} for c in all_chunks],
)

print(f"Stored {len(all_chunks)} chunks in ChromaDB at '{DB_PATH}'")

Stored 9 chunks in ChromaDB at './chroma_db_pdf'


In [20]:
## 8. Retrieval

def retrieve(question, k=TOP_K):
    query_embedding = embed_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=query_embedding, n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    chunks = []
    for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = 1 - dist
        chunks.append({"text": text, "page": meta["page"], "score": similarity})
    return chunks

def retrieval_confidence_ok(chunks):
    if not chunks:
        return False
    return chunks[0]["score"] >= MIN_SIMILARITY_THRESHOLD

test_results = retrieve("What is this document about?")
for r in test_results:
    print(f"  page {r['page']} | score {r['score']:.2f} | {r['text'][:80]}...")

  page 1 | score -0.90 | dark corner. Direct sunlight through a window can scorch the leaves of most trop...
  page 2 | score -0.98 | their active growing season.
Common Problems
Yellowing leaves are most often a s...
  page 3 | score -0.99 | Fiddle Leaf Fig: needs bright indirect light, sensitive to low humidity, drops l...
  page 1 | score -1.00 | watering next.
The Snake Plant only needs watering once every 2 to 3 weeks, sinc...


In [21]:
## 9. Prompt Construction & Generation

PROMPT_TEMPLATE = """Answer the question using ONLY the context below.
If the answer is not contained in the context, say "I don't know based on the provided document."
Cite the page number(s) you used.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, chunks):
    context = "\n\n".join(f"[Page {c['page']}]\n{c['text']}" for c in chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)

def generate(prompt):
    response = ollama.generate(model=LLM_MODEL, prompt=prompt)
    return response["response"]

def ask(question):
    chunks = retrieve(question)
    if not retrieval_confidence_ok(chunks):
        return "I don't have relevant information in this document to answer that confidently."
    prompt = build_prompt(question, chunks)
    print("[retrieved]:", ", ".join(f"page {c['page']} ({c['score']:.2f})" for c in chunks))
    return generate(prompt)

print(ask("What is this document about?"))

I don't have relevant information in this document to answer that confidently.


In [22]:
while True:
    question = input("You: ").strip()
    if question.lower() in {"exit", "quit"}:
        break
    if not question:
        continue
    answer = ask(question)
    print(f"\nAssistant: {answer}\n")

You:  how often should i water a snake plant?


[retrieved]: page 1 (0.32), page 3 (0.31), page 2 (0.04), page 3 (-0.21)

Assistant: You should water a Snake Plant every 2 to 3 weeks. (Page 1)



You:  How often should houseplants typically be repotted?


[retrieved]: page 2 (0.54), page 3 (0.28), page 1 (-0.07), page 2 (-0.19)

Assistant: Houseplants typically need repotting every 12 to 18 months, or when roots begin growing out of the drainage holes at the bottom of the pot. (Page 2)

I didn't use any page other than Page 2 for this answer.



You:  Should I fertilize my houseplants in winter?


...

You:  My plant's leaves are turning brown at the tips, what's wrong?



Assistant: I don't have relevant information in this document to answer that confidently.



You:  /



Assistant: I don't have relevant information in this document to answer that confidently.



You:  exit


In [23]:
GOLD_SET = [
    # {"question": "...", "expected_keyword": "..."},
]

if not GOLD_SET:
    print("Add a few question/expected_keyword pairs above to run an evaluation.")
else:
    hits = 0
    for case in GOLD_SET:
        answer = ask(case["question"])
        ok = case["expected_keyword"].lower() in answer.lower()
        hits += int(ok)
        print(f"{'PASS' if ok else 'FAIL'} | {case['question']}")
    print(f"\nAccuracy: {hits}/{len(GOLD_SET)} ({100*hits/len(GOLD_SET):.0f}%)")

Add a few question/expected_keyword pairs above to run an evaluation.
